In [ ]:
#A2C Rebalacing every twenty steps 


import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque
import random

sns.set()

# Parameters
initial_money = 10000
window_size = 30
commission = 0.00125
alpha_threshold = 0.004
beta_threshold = -0.004
min_trade_profit = 0.002
volatility_buffer = 0.008
trend_confirm_window = 10
rebalance_interval = 20  # rebalance every 20 time steps

# Result containers
portfolio_results = {}
dynamic_weights_log = {}
portfolio_cash = {}
shares_held = {}

# Ticker info
tickers = {
    "U11.SI": "UOB Bank",
    "C38U.SI": "CapitaLand Integrated Commercial Trust",
    "Q0F.SI": "IHH Healthcare",
    "S68.SI": "SGX",
    "S63.SI": "ST Engineering",
    "AJBU.SI": "Keppel DC REIT"
}

# Functions for technical indicators
def RMA(series, period):
    rma = [series[0]]
    alpha = 1 / period
    for price in series[1:]:
        rma.append((1 - alpha) * rma[-1] + alpha * price)
    return np.array(rma)

def apply_second_order_rma(prices):
    return RMA(RMA(prices, 25), 9)

def compute_macd(prices, short=12, long=26, signal=9):
    ema_short = pd.Series(prices).ewm(span=short).mean()
    ema_long = pd.Series(prices).ewm(span=long).mean()
    macd_line = ema_short - ema_long
    signal_line = macd_line.ewm(span=signal).mean()
    return macd_line.values, signal_line.values

def compute_rsi(prices, period=14):
    delta = np.diff(prices)
    up = np.where(delta > 0, delta, 0)
    down = np.where(delta < 0, -delta, 0)
    avg_gain = pd.Series(up).rolling(window=period).mean()
    avg_loss = pd.Series(down).rolling(window=period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return np.concatenate([np.full(period, np.nan), rsi[period:]])

def compute_volatility(prices):
    returns = np.diff(prices) / prices[:-1]
    return np.std(returns[-10:])

def is_uptrend(prices):
    trend = np.polyfit(range(len(prices)), prices, 1)[0]
    return trend > 0

# A2C components
class Actor(tf.keras.Model):
    def __init__(self, input_size, output_size, size_layer):
        super(Actor, self).__init__()
        self.lstm = tf.keras.layers.LSTM(size_layer, return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(output_size)

    def call(self, inputs, hidden_state):
        rnn_output, h, c = self.lstm(inputs, initial_state=hidden_state)
        logits = self.dense(rnn_output[:, -1])
        return logits, (h, c)

class Critic(tf.keras.Model):
    def __init__(self, input_size, size_layer):
        super(Critic, self).__init__()
        self.lstm = tf.keras.layers.LSTM(size_layer, return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(1)

    def call(self, inputs, hidden_state):
        rnn_output, h, c = self.lstm(inputs, initial_state=hidden_state)
        value = self.dense(rnn_output[:, -1])
        return value, (h, c)

class A2CAgent:
    def __init__(self, state_size, action_size, size_layer=256):
        self.state_size = state_size
        self.action_size = action_size
        self.size_layer = size_layer
        self.actor = Actor(state_size, action_size, size_layer)
        self.critic = Critic(state_size, size_layer)
        self.actor_optimizer = tf.keras.optimizers.Adam(0.001)
        self.critic_optimizer = tf.keras.optimizers.Adam(0.001)
        self.memory = deque(maxlen=1000)

    def get_state(self, t, trend):
        d = t - window_size + 1
        block = trend[d:t+1] if d >= 0 else -d * [trend[0]] + trend[0:t+1]
        delta = [block[i+1] - block[i] for i in range(window_size - 1)]
        rma2 = apply_second_order_rma(block)[-1]
        macd_line, signal_line = compute_macd(block)
        rsi = compute_rsi(block)
        macd_diff = macd_line[-1] - signal_line[-1] if len(macd_line) else 0
        rsi_val = rsi[-1] if not np.isnan(rsi[-1]) else 50
        return np.array(delta + [rma2, macd_diff, rsi_val])

    def get_action(self, state):
        state = np.array(state).reshape(1, self.state_size, 1)
        hidden = [tf.zeros((1, self.size_layer)), tf.zeros((1, self.size_layer))]
        logits, _ = self.actor(state, hidden)
        return tf.argmax(logits[0]).numpy()

# Load all price data and initialize
price_data = {}
for ticker in tickers:
    df = pd.read_csv(f"/home/priya/Desktop/Multi-agent-fuzzy-neural-stock-prediction/Agents/data_store/{ticker}.csv")
    close = df['Close'].dropna().values.tolist()
    if len(close) < window_size + 20:
        print(f"Skipping {ticker} due to insufficient data.")
        continue
    price_data[ticker] = close
    portfolio_cash[ticker] = initial_money / len(tickers)
    shares_held[ticker] = 0
    dynamic_weights_log[ticker] = {'time': [], 'macd_weight': [], 'rsi_weight': [], 'trend_weight': []}

# Run simulation
agents = {ticker: A2CAgent(window_size - 1 + 3, 3) for ticker in price_data}

for t in range(window_size, min(len(p) for p in price_data.values()) - 1):
    for ticker in price_data:
        close = price_data[ticker]
        agent = agents[ticker]  # use existing agent
        state = agent.get_state(t, close)
        
        macd_line, signal_line = compute_macd(close[max(0, t-50):t+1])
        macd_diff = macd_line[-1] - signal_line[-1] if len(macd_line) else 0
        rsi = compute_rsi(close[max(0, t-50):t+1])
        rsi_val = rsi[-1] if not np.isnan(rsi[-1]) else 50
        vol = compute_volatility(close[max(0, t-20):t+1])
        trend_score = 1 if is_uptrend(close[max(0, t-trend_confirm_window):t+1]) else -1

        macd_score = macd_diff / (abs(macd_diff) + 1e-6)
        rsi_score = ((50 - rsi_val) / 50)
        trend_score_norm = trend_score

        total_signal_strength = abs(macd_score) + abs(rsi_score) + abs(trend_score_norm)
        w_macd = abs(macd_score) / total_signal_strength
        w_rsi = abs(rsi_score) / total_signal_strength
        w_trend = abs(trend_score_norm) / total_signal_strength

        weighted_signal = w_macd * macd_score + w_rsi * rsi_score + w_trend * trend_score_norm

        dynamic_weights_log[ticker]['time'].append(t)
        dynamic_weights_log[ticker]['macd_weight'].append(w_macd)
        dynamic_weights_log[ticker]['rsi_weight'].append(w_rsi)
        dynamic_weights_log[ticker]['trend_weight'].append(w_trend)

        current_price = close[t]
        expected_profit = close[t+1] * (1 - commission) - current_price * (1 + commission)

        action = 0
        if weighted_signal > alpha_threshold and expected_profit > min_trade_profit * current_price and vol > volatility_buffer:
            action = 1
        elif weighted_signal < beta_threshold and shares_held[ticker] > 0 and vol > volatility_buffer:
            action = 2

        if action == 1 and portfolio_cash[ticker] >= current_price * (1 + commission):
            shares_to_buy = portfolio_cash[ticker] // (current_price * (1 + commission))
            portfolio_cash[ticker] -= shares_to_buy * current_price * (1 + commission)
            shares_held[ticker] += shares_to_buy

        elif action == 2 and shares_held[ticker] > 0:
            portfolio_cash[ticker] += shares_held[ticker] * current_price * (1 - commission)
            shares_held[ticker] = 0

    if (t - window_size) % rebalance_interval == 0 and t != window_size:
        total_value = sum(
            portfolio_cash[ticker] + shares_held[ticker] * price_data[ticker][t] for ticker in price_data
        )
        equal_cash = total_value / len(price_data)
        for ticker in price_data:
            portfolio_cash[ticker] = equal_cash
            shares_held[ticker] = 0


# Final ROI per ticker
for ticker in price_data:
    final_price = price_data[ticker][-1]
    final_value = portfolio_cash[ticker] + shares_held[ticker] * final_price
    roi = (final_value - (initial_money / len(price_data))) / (initial_money / len(price_data)) * 100
    portfolio_results[ticker] = {"ROI": roi, "Final Value": final_value}

results_df = pd.DataFrame(portfolio_results).T
print(results_df)


2025-05-08 20:12:52.950571: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


                ROI  Final Value
U11.SI   180.768423  4679.473716
C38U.SI  183.113244  4718.554073
Q0F.SI   177.498094  4624.968227
S68.SI   180.768423  4679.473716
S63.SI   259.449986  5990.833107
AJBU.SI  185.633222  4760.553695


Simulation Loop
At each time step t:

Each agent evaluates the current state using:

MACD diff

RSI value

Trend slope

Volatility

Calculates dynamic weights for each signal based on their strength.

Computes weighted signal score.

Decision Rules:

Buy if signal is strongly positive and profit expectation + volatility exceed thresholds.

Sell if signal is strongly negative and shares are held.

Update cash and share holdings based on trade outcomes.

Rebalancing:

Every rebalance_interval, reset holdings and split total portfolio value equally across tickers.